> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 2 · Notebook 05 — Stationarity and mean reversion

**Sessions:** S5 (Stationarity, autocorrelation & mean reversion) · [Lesson plan](../../docs/lessons/PART_02_QUANT_TOOLKIT.md)

**You will:**
1. Test prices and returns for stationarity (ADF and KPSS together).
2. Measure mean reversion with the variance ratio and half-life.
3. See a spurious regression happen.

How these notebooks work: the loading and plotting code is written for you. Cells marked **✍️ Your turn** need 1–5 lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p2lib.py is in notebooks/part02/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p2lib as p

p.use_course_style()
pd.set_option("display.float_format", "{:,.4f}".format)
prices = p.load_prices()          # dates × 10 tickers (course data via P2_DATA, else synthetic)
rets = p.log_returns(prices)      # daily log returns
prices.tail(3)

## 1. Unit-root tests: ADF (null = unit root) and KPSS (null = stationary)

In [ ]:
import warnings
from statsmodels.tsa.stattools import adfuller, kpss
spread = np.log(prices["XOM"]) - 0.9 * np.log(prices["XLE"])      # a candidate mean-reverting spread
series = {"SPY log price": np.log(prices["SPY"]), "SPY log return": rets["SPY"], "XOM − 0.9·XLE spread": spread}
rows = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                                 # KPSS warns when p is outside its table
    for name, s in series.items():
        rows[name] = {"ADF p-value": adfuller(s.dropna())[1], "KPSS p-value": kpss(s.dropna(), nlags="auto")[1]}
pd.DataFrame(rows).T

## 2. Variance ratio and half-life

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
q = 5
lp = np.log(prices["SPY"]).to_numpy()
r1, rq = np.diff(lp), lp[q:] - lp[:-q]
vr_spy = rq.var(ddof=1) / (q * r1.var(ddof=1))
vr_spy = p.check("variance ratio (SPY)", vr_spy, p.variance_ratio(lp, q))
print(f"SPY VR = {vr_spy:.3f} (≈1 random walk);  spread VR = {p.variance_ratio(spread.to_numpy(), q):.3f}")

In [ ]:
x = spread.to_numpy()
b, a = np.polyfit(x[:-1], x[1:], 1)
hl = np.log(2) / -np.log(b)
hl = p.check("spread half-life (days)", hl, p.half_life(x))

In [ ]:
z = (spread - spread.rolling(60).mean()) / spread.rolling(60).std()
ax = z.plot(title=f"Spread z-score (60-day window); half-life ≈ {hl:.1f} days")
for lvl in (-2, 2): ax.axhline(lvl, color="#8a8984", lw=1, ls="--")
ax.axhline(0, color="#52514e", lw=1); ax.set_xlabel(""); plt.show()

## 3. Spurious regression

Regress one random walk on another, independent one.

In [ ]:
import statsmodels.api as sm
rng = np.random.default_rng(3)
a_walk, b_walk = np.cumsum(rng.normal(size=1000)), np.cumsum(rng.normal(size=1000))
levels = sm.OLS(a_walk, sm.add_constant(b_walk)).fit()
diffs = sm.OLS(np.diff(a_walk), sm.add_constant(np.diff(b_walk))).fit()
print(f"Levels:      t = {levels.tvalues[1]:6.1f},  R² = {levels.rsquared:.2f}   <- 'significant' but meaningless")
print(f"Differences: t = {diffs.tvalues[1]:6.1f},  R² = {diffs.rsquared:.3f}")

## 4. Can an AR model beat a zero forecast out of sample?

In [ ]:
split = int(len(rets) * 0.7)
train, test = rets["SPY"].iloc[:split], rets["SPY"].iloc[split:]
phi, c = np.polyfit(train.iloc[:-1], train.iloc[1:], 1)
forecast = c + phi * test.shift(1).dropna()
actual = test.iloc[1:]
print(f"AR(1) coefficient {phi:.3f}")
print(f"Out-of-sample MSE: AR(1) {np.mean((actual - forecast)**2):.3e}  vs  zero forecast {np.mean(actual**2):.3e}")

## Questions
1. Why use ADF and KPSS together? What does it mean when both reject?
2. How confident are you in the half-life estimate? (Try the first and second half of the sample.)
3. Name a real-world pair of price series that would give a spurious regression.